In [5]:
import os
import sys

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

os.environ["PYSPARK_SUBMIT_ARGS"] = (
    "--driver-memory 6g pyspark-shell"
)

from pyspark.ml.recommendation import ALS
from pyspark.ml.evaluation import RegressionEvaluator
from time import perf_counter

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .master("local[4]")
    .appName("als_ml32m")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

print("Master:", spark.sparkContext.master)
print(
    "Driver memory:",
    spark.sparkContext.getConf().get(
        "spark.driver.memory",
        "not configured",
    ),
)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/07/30 21:21:30 WARN Utils: Your hostname, DESKTOP-MU43GAL, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/07/30 21:21:30 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/home/anna/projects/ccdpp-pyspark-movielens/.venv/lib/python3.12/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/07/30 21:21:31 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/07/30 21:21:33 WARN Util

Master: local[4]
Driver memory: 6g


split

In [3]:
from pyspark import StorageLevel
train = (
    spark.read
    .parquet("data/processed/ml-32m/train")
    .persist(StorageLevel.MEMORY_AND_DISK)
)

test = (
    spark.read
    .parquet("data/processed/ml-32m/test_clean")
    .persist(StorageLevel.MEMORY_AND_DISK)
)

train_count = train.count()
test_count = test.count()

print("Train rows:", train_count)
print("Test rows:", test_count)

Train rows: 28801756
Test rows: 3196413


In [4]:
import time
from pyspark.sql import SparkSession


def stop_spark():
    global spark
    spark.catalog.clearCache()
    spark.stop()
    spark = None
    time.sleep(2)


def create_spark_session(num_cores):
    spark = (
        SparkSession.builder
        .master(f"local[{num_cores}]")
        .appName(f"als_ml1m_{num_cores}_cores")
        .config("spark.pyspark.python", sys.executable)
        .config("spark.pyspark.driver.python", sys.executable)
        .getOrCreate()
    )

    spark.sparkContext.setLogLevel("WARN")

    return spark


def create_als():
    return ALS(
        userCol="user_id",
        itemCol="item_id",
        ratingCol="rating",
        rank=5,
        regParam=0.1,
        maxIter=10,
        numUserBlocks=4,
        numItemBlocks=4,
        coldStartStrategy="drop",
        seed=42,
    )

spark = create_spark_session(4)

26/07/30 21:22:26 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [6]:
core_results = []
CORE_VALUES = [1, 2, 4]
NUM_RUNS = 2
for num_cores in CORE_VALUES:
    print(f"\n{num_cores} core")


    stop_spark()
    spark = create_spark_session(num_cores)

    print(
        "Default parallelism:",
        spark.sparkContext.defaultParallelism,
    )

    train_df = (
        spark.read
        .parquet("data/processed/ml-32m/train")
        .cache()
    )

    test_df = (
        spark.read
        .parquet("data/processed/ml-32m/test_clean")
        .cache()
    )


    train_count = train_df.count()
    test_count = test_df.count()

    print("Train rows:", train_count)
    print("Test rows:", test_count)

    als = create_als()

    evaluator = RegressionEvaluator(
        labelCol="rating",
        predictionCol="prediction",
        metricName="rmse",
    )


    _ = als.fit(train_df)

    for run in range(1, NUM_RUNS + 1):
        print(f"run {run}/{NUM_RUNS}")

        start = perf_counter()
        model = als.fit(train_df)
        training_time = perf_counter() - start

        predictions = (
            model.transform(test_df)
            .cache()
        )

        prediction_count = predictions.count()
        rmse = evaluator.evaluate(predictions)
        coverage = prediction_count / test_count

        core_results.append(
            {
                "dataset": "ml-1m",
                "algorithm": "ALS",
                "num_cores": num_cores,
                "run": run,
                "train_count": train_count,
                "test_count": test_count,
                "prediction_count": prediction_count,
                "training_time": training_time,
                "rmse": rmse,
                "coverage": coverage,
                "rank": 5,
                "reg_param": 0.1,
                "max_iter": 10,
                "num_user_blocks": 4,
                "num_item_blocks": 4,
            }
        )

        print(
            f"time={training_time:.3f}s, "
            f"RMSE={rmse:.6f}, "
            f"coverage={coverage:.6f}"
        )

        predictions.unpersist()

    train_df.unpersist()
    test_df.unpersist()


1 core


26/07/30 21:23:32 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/07/30 21:23:32 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


Default parallelism: 1


Train rows: 28801756
Test rows: 3196413


run 1/2


...

Default parallelism: 4


Train rows: 28801756
Test rows: 3196413


run 1/2


time=127.601s, RMSE=0.813219, coverage=1.000000
run 2/2


time=125.517s, RMSE=0.813219, coverage=1.000000


In [7]:
core_results_df = spark.createDataFrame(core_results)

core_results_df.select(
    "num_cores",
    "run",
    "training_time",
    "rmse",
    "coverage",
).orderBy(
    "num_cores",
    "run",
).show(truncate=False)

+---------+---+------------------+------------------+--------+
|num_cores|run|training_time     |rmse              |coverage|
+---------+---+------------------+------------------+--------+
|1        |1  |439.38814559100047|0.8132186189489732|1.0     |
|1        |2  |399.69696906399986|0.8132186189489732|1.0     |
|2        |1  |263.76688825699966|0.8132186189489732|1.0     |
|2        |2  |201.58106560800115|0.8132186189489732|1.0     |
|4        |1  |127.60065187400141|0.8132186189489732|1.0     |
|4        |2  |125.51698431399927|0.8132186189489732|1.0     |
+---------+---+------------------+------------------+--------+



In [8]:
core_summary = (
    core_results_df
    .groupBy("num_cores")
    .agg(
        F.avg("training_time").alias("mean_time"),
        F.stddev("training_time").alias("std_time"),
        F.avg("rmse").alias("mean_rmse"),
        F.stddev("rmse").alias("std_rmse"),
        F.avg("coverage").alias("mean_coverage"),
        F.count("*").alias("num_runs"),
    )
)

In [9]:
baseline_time = (
    core_summary
    .filter(F.col("num_cores") == 1)
    .select("mean_time")
    .first()[0]
)

core_summary = (
    core_summary
    .withColumn(
        "speedup",
        F.lit(baseline_time) / F.col("mean_time"),
    )
    .withColumn(
        "efficiency",
        F.col("speedup") / F.col("num_cores"),
    )
    .orderBy("num_cores")
)

core_summary.show(truncate=False)

+---------+------------------+------------------+------------------+--------+-------------+--------+------------------+------------------+
|num_cores|mean_time         |std_time          |mean_rmse         |std_rmse|mean_coverage|num_runs|speedup           |efficiency        |
+---------+------------------+------------------+------------------+--------+-------------+--------+------------------+------------------+
|1        |419.54255732750016|28.06590007551445 |0.8132186189489732|0.0     |1.0          |2       |1.0               |1.0               |
|2        |232.6739769325004 |43.97201688877084 |0.8132186189489732|0.0     |1.0          |2       |1.8031348535775922|0.9015674267887961|
|4        |126.55881809400034|1.4733754614159411|0.8132186189489732|0.0     |1.0          |2       |3.3150005953428625|0.8287501488357156|
+---------+------------------+------------------+------------------+--------+-------------+--------+------------------+------------------+



In [10]:
(
    core_results_df
    .write
    .mode("overwrite")
    .parquet("results/ml-32m/als_core_scaling_runs")
)

In [11]:
(
    core_summary
    .write
    .mode("overwrite")
    .parquet("results/ml-32m/als_core_scaling_summary")
)

In [12]:
(
    core_summary
    .coalesce(1)
    .write
    .mode("overwrite")
    .option("header", True)
    .csv("results/ml-32m/als_core_scaling_summary_csv")
)

In [ ]:
core_results_df.select(
    "num_cores",
    "run",
    "training_time",
).orderBy(
    "num_cores",
    "run",
).show(truncate=False)

In [17]:
spark.stop()